<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W9D4_MiniProjet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import asyncio
import nest_asyncio
import textwrap
from pathlib import Path

nest_asyncio.apply()

# -----------------------------
# 3. Gemini API Key
# -----------------------------
# Replace with your own key
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"

# -----------------------------
# 4. Confirm Node/NPM
# -----------------------------
!node --version
!npx --version

# -----------------------------
# 5. Create working directory
# -----------------------------
WORKDIR = "/content/research_workspace"

Path(WORKDIR).mkdir(exist_ok=True)

Path(f"{WORKDIR}/ai_notes.txt").write_text("""
Artificial Intelligence enables machines to perform tasks that usually require human intelligence.

Machine Learning is a subset of AI.

Deep Learning is a subset of Machine Learning.
""")

# -----------------------------
# 6. Create Custom MCP Server
# -----------------------------
server_path = Path("/content/custom_mcp_server.py")

server_path.write_text(textwrap.dedent("""
from fastmcp import FastMCP
from typing import Dict, List

mcp = FastMCP(name="custom_ops")

@mcp.tool
def ping() -> str:
    return "pong"

@mcp.tool
def summarize_lines(lines: List[str]) -> Dict[str, int]:
    total = len(lines)
    nonempty = sum(1 for line in lines if line.strip())

    return {
        "total_lines": total,
        "nonempty_lines": nonempty
    }

@mcp.tool
def count_words(text: str) -> int:
    return len(text.split())

if __name__ == "__main__":
    mcp.run(transport="stdio")
"""))

print("Custom server created.")

# -----------------------------
# 7. MCP Connections
# -----------------------------
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {

    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": [
            "-y",
            "@modelcontextprotocol/server-filesystem",
            WORKDIR
        ],
    },

    "git": {
        "transport": "stdio",
        "command": "python",
        "args": [
            "-m",
            "mcp_server_git",
            "--repository",
            WORKDIR,
        ],
    },

    "custom_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_path)],
    },
}

client = MultiServerMCPClient(
    mcp_connections,
    tool_name_prefix=True
)

# -----------------------------
# 8. Load MCP Tools
# -----------------------------
tools = asyncio.get_event_loop().run_until_complete(
    client.get_tools()
)

print(f"Loaded {len(tools)} tools")

for t in tools:
    print("-", t.name)

# -----------------------------
# 9. Create Gemini Model
# -----------------------------
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0
)

# -----------------------------
# 10. Build Agent
# -----------------------------
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a research assistant.\n\nUse available MCP tools whenever useful.\n\nYou can:\n- inspect files\n- use git tools\n- summarize content\n- analyse text\n"""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

agent = create_tool_calling_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# -----------------------------
# 11. Test the Agent
# -----------------------------
response = agent_executor.invoke({
    "input":
    """
    Read the research file in the workspace.

    Then:
    1. Count how many non-empty lines exist.
    2. Summarize the content.
    3. Return a short report.
    """
})

print("\nFINAL ANSWER:\n")
print(response["output"])

v20.19.0
10.8.2
Custom server created.


UnsupportedOperation: fileno